# Apple News — FinBERT Sentiment Analysis

Inference on collected Google News RSS **headlines** using `ProsusAI/finbert`. This notebook does not train FinBERT. Run after `apple_news_past_year_browser_recovery.ipynb`.

## 1. Setup
Install in your selected Jupyter environment if needed: `python -m pip install torch transformers pandas tqdm`. First run downloads the model; internet required.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

# Works when notebook is run from the project root or its notebooks/ folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INPUT_CSV = RAW_DIR / "aapl_news_2025_2026.csv"
OUTPUT_CSV = RAW_DIR / "aapl_news_finbert_2025_2026.csv"
DAILY_CSV = RAW_DIR / "aapl_daily_sentiment_2025_2026.csv"
MODEL_ID = "ProsusAI/finbert"
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)
print("Input:", INPUT_CSV)
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Missing {INPUT_CSV}. Run news scraping first or adjust PROJECT_ROOT.")

Device: cpu
Input: /Users/my/Desktop/TWO-LSTM/FinBert_LSTMGRU_GAN/finbert/data/raw/aapl_news_2025_2026.csv


## 2. Read and validate news

In [8]:
df = pd.read_csv(INPUT_CSV)
required = ["Article Title", "Article URL", "Published At UTC"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")
df["Article Title"] = df["Article Title"].fillna("").astype(str).str.strip()
df["Article URL"] = df["Article URL"].fillna("").astype(str).str.strip()
df["Published At UTC"] = pd.to_datetime(df["Published At UTC"], utc=True, errors="coerce")
df = (df.loc[df["Article Title"].ne("") & df["Published At UTC"].notna()]
        .drop_duplicates(subset=["Article URL"])
        .sort_values("Published At UTC").reset_index(drop=True))
if df.empty:
    raise ValueError("No valid headlines to score.")
print("Headlines:", len(df), "| Range:", df["Published At UTC"].min(), "to", df["Published At UTC"].max())
display(df[required].head())

Headlines: 795 | Range: 2025-10-10 07:00:00+00:00 to 2026-10-09 14:35:42+00:00


,Article Title,Article URL,Published At UTC
0,Apple Accused of AI Copyright Infringement by ...,https://news.google.com/rss/articles/CBMinwFBV...,2025-10-10 07:00:00+00:00
1,Apple sued over use of copyrighted books to tr...,https://news.google.com/rss/articles/CBMi1AFBV...,2025-10-10 07:00:00+00:00
2,Key facts: Apple Inc. ranks in top 10 S&P 500 ...,https://news.google.com/rss/articles/CBMi4gFBV...,2025-10-11 07:00:00+00:00
3,Neuroscientists Sue Apple Over Pirated 'Books3...,https://news.google.com/rss/articles/CBMiqAFBV...,2025-10-13 07:00:00+00:00
4,Apple to Open iPhone Air Preorders in China on...,https://news.google.com/rss/articles/CBMipwFBV...,2025-10-13 07:00:00+00:00


## 3. Load pretrained FinBERT
Use model label mapping instead of assuming a fixed class order.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()
print("Label mapping:", model.config.id2label)
labels = [str(model.config.id2label[i]).lower() for i in range(model.config.num_labels)]
if set(labels) != {"positive", "negative", "neutral"}:
    raise ValueError(f"Unexpected FinBERT labels: {labels}")

Label mapping: {0: 'positive', 1: 'negative', 2: 'neutral'}


## 4. Predict sentiment in batches
Scores are based on **titles only**, not full article bodies.

In [10]:
all_probs = []
with torch.inference_mode():
    for start in tqdm(range(0, len(df), BATCH_SIZE), desc="FinBERT"):
        titles = df["Article Title"].iloc[start:start+BATCH_SIZE].tolist()
        inputs = tokenizer(titles, return_tensors="pt", padding=True, truncation=True, max_length=256)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        logits = model(**inputs).logits
        all_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
probs = np.vstack(all_probs)
for i, label in enumerate(labels):
    df[f"P_{label}"] = probs[:, i]
df["Sentiment Label"] = [labels[i] for i in probs.argmax(axis=1)]
df["Score"] = df["P_positive"] - df["P_negative"]
print("Scored:", len(df))
display(df[["Article Title", "P_positive", "P_negative", "P_neutral", "Sentiment Label", "Score"]].head(10))

FinBERT:   0%|          | 0/50 [00:00<?, ?it/s]

Scored: 795


,Article Title,P_positive,P_negative,P_neutral,Sentiment Label,Score
0,Apple Accused of AI Copyright Infringement by ...,0.030692,0.630461,0.338847,negative,-0.599769
1,Apple sued over use of copyrighted books to tr...,0.021388,0.729345,0.249267,negative,-0.707957
2,Key facts: Apple Inc. ranks in top 10 S&P 500 ...,0.479328,0.013359,0.507313,neutral,0.465969
3,Neuroscientists Sue Apple Over Pirated 'Books3...,0.011946,0.935975,0.052079,negative,-0.924029
4,Apple to Open iPhone Air Preorders in China on...,0.062758,0.014108,0.923134,neutral,0.048649
5,Apple expands renewable energy projects across...,0.792444,0.010059,0.197497,positive,0.782385
6,Apple unveils new 14‑inch MacBook Pro powered ...,0.554808,0.015740,0.429452,positive,0.539068
7,"Apple unleashes M5, the next big leap in AI pe...",0.218494,0.014099,0.767406,neutral,0.204395
8,"Apple Updates iPad Pro, MacBook Pro, Vision Pr...",0.053221,0.016349,0.930430,neutral,0.036872
9,Apple introduces the powerful new iPad Pro wit...,0.267446,0.011796,0.720758,neutral,0.255650


## 5. Save article-level results

In [11]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)
print(df["Sentiment Label"].value_counts())

Saved: /Users/my/Desktop/TWO-LSTM/FinBert_LSTMGRU_GAN/finbert/data/raw/aapl_news_finbert_2025_2026.csv
Sentiment Label
neutral     525
negative    139
positive    131
Name: count, dtype: int64


## 6. Optional: daily summary for exploration
**Important:** This is a UTC-calendar-day aggregation, **not** a trading-day-aligned feature. For prediction, shift news according to market close, weekends, and the forecast target; do not merge it directly with same-day closing prices.

In [12]:
df["News Date UTC"] = df["Published At UTC"].dt.date
by_day = (df.groupby("News Date UTC", as_index=False)
          .agg(Score=("Score", "mean"),
               News_Count=("Score", "size"),
               Positive_Mean=("P_positive", "mean"),
               Negative_Mean=("P_negative", "mean"),
               Neutral_Mean=("P_neutral", "mean")))
by_day.to_csv(DAILY_CSV, index=False)
print("Saved daily summary:", DAILY_CSV)
display(by_day.tail(10))

Saved daily summary: /Users/my/Desktop/TWO-LSTM/FinBert_LSTMGRU_GAN/finbert/data/raw/aapl_daily_sentiment_2025_2026.csv


,News Date UTC,Score,News_Count,Positive_Mean,Negative_Mean,Neutral_Mean
253,2026-09-30,-0.208886,4,0.037707,0.246593,0.715699
254,2026-10-01,-0.240875,4,0.021408,0.262283,0.716309
255,2026-10-02,-0.000915,20,0.158981,0.159896,0.681123
256,2026-10-03,-0.148020,10,0.063707,0.211727,0.724566
257,2026-10-04,0.065401,13,0.095005,0.029604,0.875391
258,2026-10-05,-0.105221,10,0.032944,0.138166,0.828890
259,2026-10-06,-0.112126,7,0.041135,0.153262,0.805603
260,2026-10-07,-0.091941,12,0.078831,0.170771,0.750398
261,2026-10-08,0.127078,11,0.266482,0.139404,0.594114
262,2026-10-09,-0.323609,13,0.040279,0.363887,0.595834


## Research checks
- Verify scraper coverage CSV: missing months mean missing features, **not** neutral sentiment.
- Google News RSS may cap results; headline sample is not guaranteed complete.
- Do not fit price scalers on validation/test periods.
- For forecasting, construct features available **before** the prediction cutoff, with timezone-aware market calendars.
- This score formula may differ from your original `News.csv`; compare definitions before combining experiments.